# Looped transformers: shared weights

Reusing a core, computation depth, and backpropagation through loops.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A looped transformer applies the same core repeatedly. Unique parameter count stays fixed while computation and activation memory generally grow. A list of independent block copies is not weight sharing.

This exercise reuses a causal-attention core. Full backprop traverses every iteration. Detaching between iterations truncates gradients and changes training; correctly applied checkpointing recomputes activations while preserving gradients. Increasing inference depth is an experiment, not a guaranteed improvement. Configure these choices on the Builder looped block.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
class Looped(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = nn.TransformerEncoderLayer(16, 4, 32, dropout=0, batch_first=True)
    def forward(self, h, loops):
        mask = torch.triu(torch.ones(h.shape[1], h.shape[1], dtype=torch.bool), diagonal=1)
        for _ in range(loops):
            h = self.core(h, src_mask=mask)
        return h
model = Looped(); x = torch.randn(2, 6, 16)
parameters = sum(p.numel() for p in model.parameters())
for depth in [1, 2, 4]:
    model.zero_grad(set_to_none=True)
    output = model(x, depth)
    # LayerNorm makes output.square().mean() nearly constant; use a nonuniform target.
    loss = F.mse_loss(output, x * 0.3)
    loss.backward()
    print('loops:', depth, 'unique parameters:', parameters, 'core gradient:', model.core.linear1.weight.grad.norm().item())

## Try it yourself

Compare with a ModuleList of four independent cores. Check parameter counts and id(weight). Why do equal initial values not establish weight sharing?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [Universal Transformers](https://arxiv.org/abs/1807.03819)
- [ALBERT](https://arxiv.org/abs/1909.11942)